In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv('/content/drive/MyDrive/Machine Learning/Projects/Fraud Detection Capstone Project/Datasets/Transaction Fraud Detection/transactions_data.csv')

df.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [5]:
df_copy = df.copy()

# Knowing the Data

- **id**: Transaction id
- **data**: Date and time of transaction
- **client_id**: Client unique id
- **card_id**: Card unique id
- **use_chip**: Type of transaction (swipe/online/chip)
- **merchant_id**: Unique id
- **merchant_city**: City of the merchant
- **merchant_state**: State of the merchant
  * where merchant_city = Online, all merchant_state is nan
- **zip**: zip no.
  * If merchant_city = Online, then zip is nan. Also there are more nan other than this.
  * Number of non-online transactions missing a zip code: 89006

- **mcc**: mcc code
- **error**: Error in transaction
  * 211393 data contain error

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 12 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              int64  
 1   date            object 
 2   client_id       int64  
 3   card_id         int64  
 4   amount          object 
 5   use_chip        object 
 6   merchant_id     int64  
 7   merchant_city   object 
 8   merchant_state  object 
 9   zip             float64
 10  mcc             int64  
 11  errors          object 
dtypes: float64(1), int64(5), object(6)
memory usage: 1.2+ GB


In [10]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)
np.set_printoptions(suppress=True)

In [11]:
df.describe()

,id,client_id,card_id,merchant_id,zip,mcc
count,13305915.00,13305915.00,13305915.00,13305915.00,11653209.00,13305915.00
mean,15584024.57,1026.81,3475.27,47723.76,51327.82,5565.44
std,4704498.65,581.64,1674.36,25815.34,29404.23,875.70
min,7475327.00,0.00,0.00,1.00,1001.00,1711.00
25%,11506044.50,519.00,2413.00,25887.00,28602.00,5300.00
50%,15570866.00,1070.00,3584.00,45926.00,47670.00,5499.00
75%,19653605.50,1531.00,4901.00,67570.00,77901.00,5812.00
max,23761874.00,1998.00,6144.00,100342.00,99928.00,9402.00


In [12]:
df.isna().sum()

,0
id,0
date,0
client_id,0
card_id,0
amount,0
use_chip,0
merchant_id,0
merchant_city,0
merchant_state,1563700
zip,1652706


In [24]:
(df['merchant_city'] == 'ONLINE').sum()

np.int64(1563700)

In [27]:
total_duplicates = df.duplicated().sum()
print(f'Total exact duplicate rows: {total_duplicates}')
potential_double_billing = df.duplicated(subset=['date', 'client_id', 'card_id', 'amount']).sum()
print(f'Potential double-billing instances: {potential_double_billing}')

Total exact duplicate rows: 0
Potential double-billing instances: 101


In [32]:
potential_duplicates_df = df[df.duplicated(subset=['date', 'client_id', 'card_id', 'amount'], keep=False)]

display(potential_duplicates_df.sort_values(by='client_id').head(20))

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
6652118,15569837,2015-01-22 08:32:00,34,3342,$60.00,Chip Transaction,27092,Sacramento,CA,95829.00,4829,NaN
6652117,15569838,2015-01-22 08:32:00,34,3342,$60.00,Chip Transaction,49789,Sacramento,CA,95829.00,5541,NaN
3694551,11954250,2012-11-18 08:54:00,34,1166,$40.00,Swipe Transaction,27092,Sacramento,CA,95829.00,4829,NaN
3694550,11954249,2012-11-18 08:54:00,34,1166,$40.00,Swipe Transaction,27092,Sacramento,CA,95829.00,4829,Insufficient Balance
3273747,11441812,2012-07-25 12:20:00,53,3267,$-93.00,Swipe Transaction,43293,Vacaville,CA,95687.00,5499,NaN
3273748,11441813,2012-07-25 12:20:00,53,3267,$-93.00,Swipe Transaction,43293,Vacaville,CA,95687.00,5499,NaN
6434509,15302646,2014-11-25 12:23:00,53,3267,$-52.00,Swipe Transaction,61195,Sutter Creek,CA,95685.00,5541,NaN
6434510,15302645,2014-11-25 12:23:00,53,3267,$-52.00,Swipe Transaction,61195,Sutter Creek,CA,95685.00,5541,NaN
684283,8294600,2010-07-23 21:14:00,87,109,$20.00,Swipe Transaction,27092,Leander,TX,78641.00,4829,Insufficient Balance
684284,8294601,2010-07-23 21:14:00,87,109,$20.00,Swipe Transaction,27092,Leander,TX,78641.00,4829,NaN


- these are likely system glitches or rapid user retries rather than a data integrity issue

### id

In [37]:
df['id'].duplicated().sum()


np.int64(0)

In [36]:
df['id'].dtype

dtype('int64')

### date

In [39]:
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d %H:%M:%S')
df['date'].head()

,date
0,2010-01-01 00:01:00
1,2010-01-01 00:02:00
2,2010-01-01 00:02:00
3,2010-01-01 00:05:00
4,2010-01-01 00:06:00


In [43]:
df.insert(2, 'time', df['date'].dt.time)
df['date'] = df['date'].dt.date

In [44]:
df.head()

,id,date,time,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01,00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.00,5499,NaN
1,7475328,2010-01-01,00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.00,5311,NaN
2,7475329,2010-01-01,00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.00,4829,NaN
3,7475331,2010-01-01,00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.00,4829,NaN
4,7475332,2010-01-01,00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.00,5813,NaN


### amount

In [45]:
df['amount'].isna().sum()

np.int64(0)

In [48]:
df['amount'] = df['amount'].str.replace("$","").str.strip().astype(float)

In [49]:
df['amount'].describe()

,amount
count,13305915.00
mean,42.98
std,81.66
min,-500.00
25%,8.93
50%,28.99
75%,63.71
max,6820.20


### use_chip

In [50]:
df['use_chip'].value_counts()

,count
use_chip,
Swipe Transaction,6967185
Chip Transaction,4780818
Online Transaction,1557912


### merchant_city

In [51]:
df['merchant_city'] = df['merchant_city'].str.strip()
df['merchant_city'] = df['merchant_city'].str.title()

### merchant_state

In [58]:
len(df[(df['merchant_state'].isna()) & (df['merchant_city'] == 'Online')])

1563700

In [56]:
df[df['merchant_state'].isna() & (df['merchant_city'] != 'Online')]

,id,date,time,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors


### zip

In [59]:
non_online_missing_zip = df[df['zip'].isna() & (df['merchant_city'] != 'Online')]
print(f"Number of non-online transactions missing a zip code: {len(non_online_missing_zip)}")
display(non_online_missing_zip.head())

Number of non-online transactions missing a zip code: 89006


,id,date,time,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
581,7476010,2010-01-01,07:53:00,1579,3830,6.51,Swipe Transaction,22204,Puerto Vallarta,Mexico,NaN,5541,NaN
1026,7476549,2010-01-01,10:03:00,363,5555,8.16,Swipe Transaction,93391,Vatican City,Vatican City,NaN,5812,NaN
1148,7476704,2010-01-01,10:37:00,363,5555,11.66,Swipe Transaction,93391,Vatican City,Vatican City,NaN,5812,NaN
1706,7477375,2010-01-01,12:50:00,1266,2478,9.77,Swipe Transaction,22204,Guadalajara,Mexico,NaN,5541,NaN
1836,7477534,2010-01-01,13:19:00,1266,2478,11.32,Swipe Transaction,75804,Guadalajara,Mexico,NaN,5411,NaN


### errors

In [60]:
df['errors'].unique()

array([nan, 'Technical Glitch', 'Bad Expiration', 'Bad Card Number',
       'Insufficient Balance', 'Bad PIN', 'Bad CVV', 'Bad Zipcode',
       'Insufficient Balance,Technical Glitch',
       'Bad PIN,Insufficient Balance', 'Bad PIN,Technical Glitch',
       'Bad Expiration,Technical Glitch',
       'Bad Card Number,Bad Expiration',
       'Bad Card Number,Insufficient Balance',
       'Bad Expiration,Insufficient Balance', 'Bad Card Number,Bad CVV',
       'Bad CVV,Technical Glitch', 'Bad Expiration,Bad CVV',
       'Bad CVV,Insufficient Balance', 'Bad Card Number,Technical Glitch',
       'Bad Zipcode,Insufficient Balance',
       'Bad Card Number,Bad Expiration,Insufficient Balance',
       'Bad Zipcode,Technical Glitch'], dtype=object)

In [61]:
df.shape[0] - df['errors'].isna().sum()

np.int64(211393)

### Final Cleaning Summary
Verification of data types and non-null values after initial cleaning steps.

In [63]:
display(df.info())

display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 13 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              int64  
 1   date            object 
 2   time            object 
 3   client_id       int64  
 4   card_id         int64  
 5   amount          float64
 6   use_chip        object 
 7   merchant_id     int64  
 8   merchant_city   object 
 9   merchant_state  object 
 10  zip             float64
 11  mcc             int64  
 12  errors          object 
dtypes: float64(2), int64(5), object(6)
memory usage: 1.3+ GB


None

,id,client_id,card_id,amount,merchant_id,zip,mcc
count,13305915.00,13305915.00,13305915.00,13305915.00,13305915.00,11653209.00,13305915.00
mean,15584024.57,1026.81,3475.27,42.98,47723.76,51327.82,5565.44
std,4704498.65,581.64,1674.36,81.66,25815.34,29404.23,875.70
min,7475327.00,0.00,0.00,-500.00,1.00,1001.00,1711.00
25%,11506044.50,519.00,2413.00,8.93,25887.00,28602.00,5300.00
50%,15570866.00,1070.00,3584.00,28.99,45926.00,47670.00,5499.00
75%,19653605.50,1531.00,4901.00,63.71,67570.00,77901.00,5812.00
max,23761874.00,1998.00,6144.00,6820.20,100342.00,99928.00,9402.00


In [64]:
df.to_csv('cleaned_transaction_data.csv', index=False)